# Setup: Prepare Test Cases

Generates four test-case PDFs (TC1–TC4) and saves them to the Unity Catalog Volume. All deployment option test notebooks read PDFs from this shared location.

**Idempotent** — skips files that already exist.

### Test Cases

| ID | Description | Pages |
|----|-------------|-------|
| TC1 | Simple single-page text | 1 |
| TC2 | Multi-page with headings and bullets | 3 |
| TC3 | Single-page with tabular data | 1 |
| TC4 | Long document (10 pages) with structured sections | 10 |

### Cluster Requirements

| Setting | Value |
|---------|-------|
| Instance | Serverless |
| Workers | 0 (single-node) or any |
| Libraries | None — installed via `%pip` below |

### Prerequisites
- Unity Catalog catalog and schema must exist (Volume is created automatically)

### Install Dependencies

`pymupdf` (imported as `fitz`) is used to programmatically create PDF files.
`pyyaml` is used to read `config.yaml` (pre-installed on Databricks MLR but not on serverless).

In [ ]:
%pip install pymupdf pyyaml

In [ ]:
import yaml, os

# Resolve project root (works both locally and on Databricks).
if "__file__" in dir():
    _root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
else:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _root = "/Workspace" + os.path.dirname(os.path.dirname(_nb))

cfg     = yaml.safe_load(open(f"{_root}/config.yaml"))
CATALOG = cfg["catalog"]
SCHEMA  = cfg["schema"]
VOLUME  = cfg["volume"]
TC_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{cfg['test_cases_subpath']}"

# UC Volumes must be created via SQL, not os.makedirs.
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

# Create the test_cases subdirectory within the Volume.
os.makedirs(TC_PATH, exist_ok=True)
print(f"Test cases path: {TC_PATH}")

### Generate Test PDFs

Each test case is a synthetically generated PDF designed to exercise different parsing scenarios:
- **TC1**: Simple single-page text — baseline sanity check
- **TC2**: 3-page document with chapter headings and bullet lists — tests multi-page handling
- **TC3**: Single-page with a table (Name / Score / Grade) — tests tabular extraction
- **TC4**: 10-page document with numbered sections — tests throughput on longer documents

In [ ]:
import fitz  # PyMuPDF

def _make_pdf(text: str) -> bytes:
    """Create a single-page PDF with the given text."""
    doc  = fitz.open()
    page = doc.new_page()
    page.insert_text((72, 72), text, fontsize=11)  # 1-inch margin
    return doc.tobytes()

def _make_multipage_pdf(pages: list) -> bytes:
    """Create a multi-page PDF with one page per text string."""
    doc = fitz.open()
    for text in pages:
        page = doc.new_page()
        page.insert_text((72, 72), text, fontsize=11)
    return doc.tobytes()

# Define test cases with varying complexity
test_cases = {
    # TC1: Simple single-page text (baseline)
    "tc1.pdf": _make_pdf(
        "Hello MinerU!\nThis is a simple test document.\nPage 1 of 1."
    ),
    # TC2: 3-page document with headings and bullet lists
    "tc2.pdf": _make_multipage_pdf([
        f"Chapter {i}: Section Title\n\nParagraph text for page {i}.\n\n"
        "- Bullet point one\n- Bullet point two\n- Bullet point three"
        for i in range(1, 4)
    ]),
    # TC3: Single-page with tabular data
    "tc3.pdf": _make_pdf(
        "Student Results\n\nName        Score   Grade\n"
        "Alice       95      A\nBob         82      B\nCharlie     76      C"
    ),
    # TC4: 10-page document with structured sections
    "tc4.pdf": _make_multipage_pdf([
        f"Section {i}: Analysis and Results\n\n"
        f"This section covers topic {i} in detail.\n\n"
        f"Key findings:\n1. Finding one\n2. Finding two\n3. Conclusion"
        for i in range(1, 11)
    ]),
}

# Write each PDF to the Volume (skip if already exists)
for fname, pdf_bytes in test_cases.items():
    path = os.path.join(TC_PATH, fname)
    if os.path.exists(path):
        print(f"  {fname}: already exists — skipping")
    else:
        with open(path, "wb") as f:
            f.write(pdf_bytes)
        print(f"  {fname}: written ({len(pdf_bytes)} bytes)")

print(f"\nTest cases ready at {TC_PATH}")